# T5-Small — DIMER text-to-text generation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/t5-small-text2text-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/t5-small-text2text-pipeline/blob/main/tutorials/t5_small_text2text_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--t5%2Ft5--small-ffcc4d?style=flat)](https://huggingface.co/google-t5/t5-small) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftext--to--text--transfer--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/text-to-text-transfer-transformer) [![arXiv](https://img.shields.io/badge/arXiv-1910.10683-b31b1b.svg)](https://arxiv.org/abs/1910.10683)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** caller-prefixed text-to-text generation (summarisation and English→German/French/Romanian translation) using the pinned `google-t5/t5-small` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/t5_small_text2text_pipeline/pipeline.py` at revision `1efacb0807bb`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `df1b051c49625cf57a3d0d8d3863ed4d13564fe4` (~244 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

`google-t5/t5-small` is the original 60 M-parameter T5 of Raffel et al. (2020) — **not FLAN-T5**, so it follows only the task prefixes it was trained on, not free-form instructions. At inference the encoder reads the whole prefixed input once and the decoder emits one SentencePiece token per step until `</s>` or a step ceiling; **greedy decoding** (per-step argmax, `num_beams=1`, `do_sample=False`) is the default decision rule and beam search is available on request — there is no sampling and no seed. The caller writes the task prefix — `summarize: `, `translate English to German: `, `translate English to French: ` or `translate English to Romanian: ` — in front of the text: **the pipeline invents no prefix** and only reports which known one an input started with (`known_prefix`, or `None`). **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the model, the SentencePiece tokenizer and the prefix convention; what the carried pipeline module adds is manifest verification, input validation with named ceilings, a fixed output contract, the list of trained prefixes as `TASK_PREFIXES`, and the `validate_inputs` and `evaluation_report` stage helpers.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author two already-prefixed inputs (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and the decision rule and validate the inputs into an input manifest before any model work, generate through the public API with explicit `max_new_tokens`/`num_beams`, read `stopped_by`, `known_prefix` and the token counts correctly, read from the machine-readable evaluation report why **no metric is reported** and what references a ROUGE/BLEU evaluation would need, and export every generation with its identifier plus provenance.

**This notebook does not demonstrate:** instruction following or chat, sampling-based decoding, batching, classification or embedding (the GLUE tasks in the training mixture are not exposed), source languages other than English or target languages other than German, French and Romanian, the upstream `task_specific_params` decoding settings (`min_length`, `length_penalty`, `no_repeat_ngram_size` are not applied by the pipeline), or any ROUGE/BLEU measurement. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float32 there too). CPU is adequate: the repository's model card records, for the Windows-venv smoke on an Intel Core Ultra 9 275HX, 3.89 s to load and digest-verify the 244 MB snapshot, 0.16 s for the 11-token translation (greedy; 0.09 s with `num_beams=4`) and 0.31 s for a 94-token `summarize: ` input that generated 48 tokens; the `t5-base-text2text-pipeline` sibling (220 M parameters, 892 MB snapshot) took 1.2× as long to load and 1.8–2.7× as long per call on the same CPU in its own smoke, as both cards record. The pinned `torch==2.14.0` install and the 242 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what greedy decoding and beam search do; why a generated sentence can be fluent and wrong.
- **Data:** the default sample is **synthetic** — two already-prefixed inputs authored in code (one translation sentence, one four-sentence passage to summarise) — so nothing is downloaded and no private data is needed. It carries no reference outputs, so any number it produces is smoke/sanity evidence, never a quality measurement. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction; each non-empty line is one input that must already start with its task prefix. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google-t5/t5-small` snapshot (~244 MB in total) at revision `df1b051c4962…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'sentencepiece==0.2.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 't5-small-text2text-pipeline',
    'repository_revision': '1efacb0807bb1dfa8a37a164d80d6a5cada4f9da',
    'embedded_module': 'src/t5_small_text2text_pipeline/pipeline.py',
    'embedded_modules': ['src/t5_small_text2text_pipeline/pipeline.py'],
    'module_sha256': '0c689c4405d3540247ff0dec1f578d12bd143bce5523fb3c7c16ad32ddc4b2f3',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/t5_small_text2text_pipeline/` @ `1efacb0807bb`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/t5_small_text2text_pipeline/pipeline.py`

In [ ]:
"""Text-to-text generation with the pinned ``google-t5/t5-small`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/t5-small/``) or, when
explicitly allowed, from the Hugging Face Hub at the pinned revision. The caller supplies the task
prefix (``summarize: ``, ``translate English to German: `` ...); this module never adds one.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "google-t5/t5-small"
MODEL_REVISION = "df1b051c49625cf57a3d0d8d3863ed4d13564fe4"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "t5-small"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

MAX_INPUT_TOKENS = 512  # ``n_positions`` in the snapshot config.json; longer inputs are rejected, not cut
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call
DEFAULT_MAX_NEW_TOKENS = 64
MAX_TEXT_CHARS = 20_000  # pre-tokenisation guard on the input string
MAX_NUM_BEAMS = 8
DECISION_RULE = "greedy argmax per decoding step (num_beams=1); beam search when num_beams > 1; no sampling"
# The four prefixes the upstream checkpoint was trained on, read from ``task_specific_params`` in the
# snapshot config.json. Reported back as ``known_prefix``; the pipeline does not prepend any of them.
TASK_PREFIXES = (
    "summarize: ",
    "translate English to German: ",
    "translate English to French: ",
    "translate English to Romanian: ",
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str that already carries its task prefix; the pipeline prepends none",
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "task_prefixes": list(TASK_PREFIXES),
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "SentencePiece encoding with no prefix added and no truncation: an input over "
        "MAX_INPUT_TOKENS is rejected with a ValueError naming the count, never cut"
    ),
}


def _check_inputs(text: Any, max_new_tokens: Any, num_beams: Any) -> str:
    """Raise TypeError/ValueError naming the first violated ceiling; return the text.

    The encoder-token ceiling is not checked here because it needs the loaded tokenizer;
    ``_check_input_tokens`` applies it inside the pipeline once the count is known.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError("text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    for name, value, ceiling in (
        ("max_new_tokens", max_new_tokens, MAX_NEW_TOKENS),
        ("num_beams", num_beams, MAX_NUM_BEAMS),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not 1 <= value <= ceiling:
            raise ValueError(f"{name} must be between 1 and {ceiling}, got {value}")
    return text


def _check_input_tokens(n_input: int) -> int:
    """The encoder-token ceiling, applied once the tokenizer has counted."""
    if n_input > MAX_INPUT_TOKENS:
        raise ValueError(f"input is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}")
    return n_input


def known_prefix(text: str) -> str | None:
    """Which trained task prefix ``text`` starts with, or ``None``; nothing is prepended."""
    return next((prefix for prefix in TASK_PREFIXES if text.startswith(prefix)), None)


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = 1,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``generate`` would: both route through
    ``_check_inputs``. ``generate`` takes one text per call, so ``texts`` is the batch the notebook
    will loop over and every entry is validated with the same settings. An input that starts with
    no trained prefix is **not** rejected — the pipeline does not refuse it either — but the
    manifest records ``known_prefix: null`` so the caller can see it. The encoder-token ceiling
    (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and is enforced inside ``generate``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_inputs(text, max_new_tokens, num_beams) for text in texts]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"input{i:02d}",
                "chars": len(text),
                "known_prefix": known_prefix(text),
            }
            for i, text in enumerate(checked)
        ],
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    The repository ships no metric helper, so the verdict is always ``not-measurable`` (EVAL9).
    ``references`` exists for interface parity with the fleet's other pipelines and is recorded in
    ``reason`` rather than scored: ROUGE and BLEU need a scorer and enough referenced items to state
    a dispersion, and manufacturing a number from a proxy such as length ratio or copy rate would
    misrepresent a plumbing check as a quality measurement.
    """
    generation = result.get("generation", {})
    supplied = references is not None
    return {
        "task": "caller-prefixed text-to-text generation (summarisation, translation)",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens "
            "and stopped_by are counts and flags, and "
            f"{generation.get('decision_rule', DECISION_RULE)} produces some token at every step "
            "with no minimum-probability cut-off and no shipped acceptance threshold"
        ),
        "sample_kind": sample_kind,
        "n_generated_tokens": int(result.get("generated_tokens", 0)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no metric helper and a generation has no ground truth here"
            + (
                "; references were supplied but no metric helper exists to score them, and one "
                "reference is not a dispersion"
                if supplied
                else "; the evaluated sample has no reference outputs"
            )
        ),
        "needs": (
            "reference outputs from the deployment domain — a reference summary per document, a "
            "reference translation per sentence — over enough items to state a dispersion, scored "
            "with the caller's own ROUGE-1/2/L or BLEU/chrF implementation, excluding or re-running "
            "outputs whose stopped_by is max_new_tokens; no proxy such as length ratio or copy rate "
            "substitutes for that"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class T5SmallText2TextPipeline:
    """``_runner(text, max_new_tokens, num_beams)`` -> ``(generated_text, generated_tokens, stopped_by)``;
    ``_count_tokens(text)`` -> encoder token count incl. EOS. Both injectable so tests run offline."""

    _runner: Callable[[str, int, int], tuple[str, int, str]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> T5SmallText2TextPipeline:
        import torch
        from transformers import T5ForConditionalGeneration, T5TokenizerFast

        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = T5TokenizerFast.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = T5ForConditionalGeneration.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        eos_id, pad_id = model.config.eos_token_id, model.config.pad_token_id

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(text: str, max_new_tokens: int, num_beams: int) -> tuple[str, int, str]:
            enc = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            ids = out[0].tolist()
            content = [t for t in ids if t not in (eos_id, pad_id)]
            stopped_by = "eos" if eos_id in ids else "max_new_tokens"
            return tokenizer.decode(content, skip_special_tokens=True), len(content), stopped_by

        return cls(runner, count_tokens, resolved_device, source)

    def _validate(self, text: Any, max_new_tokens: Any, num_beams: Any) -> int:
        text = _check_inputs(text, max_new_tokens, num_beams)
        return _check_input_tokens(self._count_tokens(text))

    def generate(
        self, text: str, *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS, num_beams: int = 1
    ) -> dict[str, Any]:
        """Run one prefixed input through encoder-decoder generation; the caller owns the task prefix."""
        n_input = self._validate(text, max_new_tokens, num_beams)
        generated, n_generated, stopped_by = self._runner(text, max_new_tokens, num_beams)
        if not isinstance(generated, str) or not isinstance(n_generated, int):
            raise RuntimeError("runner must return (str, int, str)")
        prefix = known_prefix(text)
        return {
            "text": generated,
            "generated_tokens": n_generated,
            "input_tokens": n_input,
            "stopped_by": stopped_by,
            "known_prefix": prefix,
            "generation": {
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "do_sample": False,
                "decision_rule": DECISION_RULE,
            },
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `df1b051c4962…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `T5SmallText2TextPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "t5-small",
  "modelId": "google-t5/t5-small",
  "revision": "df1b051c49625cf57a3d0d8d3863ed4d13564fe4",
  "files": [
    {
      "path": "README.md",
      "bytes": 8473,
      "sha256": "6559de584e57caf42cd2c77a23bb31d5bd28c5aa9995ee1d967b510acd88d410"
    },
    {
      "path": "config.json",
      "bytes": 1206,
      "sha256": "530e25060e3a8d5f7b0fcf53bfea9f3601165161f3b1914676e98d97cf07bcf1"
    },
    {
      "path": "generation_config.json",
      "bytes": 147,
      "sha256": "f5a1c7e2be8092018d8835128987edf0111637dd98e90599cc80310fef75d95a"
    },
    {
      "path": "model.safetensors",
      "bytes": 242043056,
      "sha256": "bd944e5f1b3ad9b70dd9d00010a517059e19265671076b8b0a4a58d9491842bc"
    },
    {
      "path": "spiece.model",
      "bytes": 791656,
      "sha256": "d60acb128cf7b7f2536e8f38a5b18a05535c9e14c7a355904270e15b0945ea86"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1389353,
      "sha256": "d2acde0d8d71dd30a711834b07781b9c89feaac33fd332f60507699282740066"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2324,
      "sha256": "d1e7146101aa96282057f374aa9c3b260fba2109e5990edb1775d6efc70ffa3c"
    }
  ],
  "totalBytes": 244236215
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = T5SmallText2TextPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: two inputs authored in this cell, each already carrying its task prefix because the pipeline invents no prefix. The first is the translation sentence the repository's card-pass smoke used (`translate English to German: The house is wonderful.`); the second is a four-sentence English passage about the C4 corpus, written here after the T5 paper's description, behind `summarize: `. Neither has a reference output — no human translation, no reference summary — so nothing in this notebook is a quality measurement; the card's smoke observation that the translation came back as `Das Haus ist wunderbar.` is one run on one machine, not an expected value this notebook asserts. The sample identity and a SHA-256 of its text are printed so an export can be tied to exactly these inputs.

Two Colab form parameters fix the generation settings for every call: `GEN_MAX_NEW_TOKENS` (default 64, the package's `DEFAULT_MAX_NEW_TOKENS`) and `NUM_BEAMS` (default 1 = greedy). They are checked against the carried module's ceilings in the next section.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file in which every non-empty line is one input **that already starts with its task prefix** (for example `translate English to French: ...`); each line must be at most `MAX_TEXT_CHARS` characters and tokenise to at most `MAX_INPUT_TOKENS` SentencePiece pieces, which the pipeline enforces by rejecting, not by truncating. The upload stays inside this runtime. If you hold reference outputs for your lines, keep them outside the notebook — Section 7 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
GEN_MAX_NEW_TOKENS = 64  # @param {type:"integer"}
NUM_BEAMS = 1  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    texts = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if not texts:
        raise ValueError(f'{sample_name}: expected at least one non-empty line, each starting with its task prefix')
    sample_kind = 'BYOD upload'
else:
    texts = [
        'translate English to German: The house is wonderful.',
        'summarize: The Colossal Clean Crawled Corpus, or C4, is a cleaned version of the April 2019 Common Crawl web snapshot. '
        'The corpus was prepared by keeping only lines that end in terminal punctuation, dropping pages with fewer than five sentences, '
        'and removing pages that contain words from a block list, placeholder text or source code. '
        'Duplicate three-sentence spans were also removed so that boilerplate does not dominate the training signal. '
        'It was used to pre-train the T5 family of models with a span-corruption denoising objective.',
    ]
    sample_name = 'synthetic_prefixed_inputs'
    sample_kind = 'synthetic (authored in this cell)'
item_ids = [f'input{index:02d}' for index in range(len(texts))]
sample_sha256 = hashlib.sha256('\n'.join(texts).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256, 'max_new_tokens': GEN_MAX_NEW_TOKENS, 'num_beams': NUM_BEAMS})
for item_id, text in zip(item_ids, texts, strict=True):
    print(f'{item_id}: {text[:110]}' + ('...' if len(text) > 110 else ''))

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `generate` applies — both route through the same private `_check_inputs` — so the text type, non-emptiness, the character ceiling `MAX_TEXT_CHARS`, `max_new_tokens` in 1..`MAX_NEW_TOKENS` and `num_beams` in 1..`MAX_NUM_BEAMS` are enforced identically. `generate` takes one text per call, so the helper validates the whole batch the notebook will loop over with the same settings and returns one **input manifest** naming the schema and ceilings, each input's identifier, character count and `known_prefix`, the settings in force, and the verdict; it is written to `outputs/t5_small_text2text_input_manifest.json`. An input that starts with no trained prefix is **not** rejected — the pipeline does not refuse it either — but `known_prefix` is `null` so a reader can see it, and the model then returns something plausible-looking rather than an error. `MAX_INPUT_TOKENS` (encoder tokens including `</s>`; the upstream `n_positions`) needs the real tokenizer and is therefore enforced inside `generate`, which **rejects with a `ValueError` naming the count, never silently cuts**; every result reports `input_tokens`. `DECISION_RULE` states the decoding rule in force (greedy per-step argmax, beam search when `num_beams > 1`, no sampling) and `TASK_PREFIXES` lists the four prefixes the checkpoint was trained on. To show what rejection looks like, the cell also validates an out-of-range `num_beams` and records the pipeline's own error message as a finding. Nothing here trims or alters the texts.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS}
print(ceilings)
print({'decision_rule': DECISION_RULE})
print({'task_prefixes': list(TASK_PREFIXES)})
input_manifest = validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, names=item_ids)
# Demonstrate rejection on a setting that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/t5_small_text2text_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
unknown = [entry['id'] for entry in input_manifest['inputs'] if entry['known_prefix'] is None]
if unknown:
    print({'warning': f'{unknown} start with no trained prefix; the model will still return text, but not a summary or translation'})
print({'token_ceiling': 'enforced by generate() with the real tokenizer; reported as input_tokens'})

## 6. Generate and read the outputs correctly

`generate(text, max_new_tokens=..., num_beams=...)` runs one prefixed input through the encoder-decoder and returns a dict: `text` (the decoded generation with special tokens removed), `generated_tokens` (decoder tokens emitted, excluding `</s>`/pad), `input_tokens` (encoder tokens including `</s>`, the number checked against `MAX_INPUT_TOKENS`), `stopped_by` (`eos` when the model ended the sequence itself, `max_new_tokens` when it hit the ceiling — such an output is cut mid-thought and should be re-run with a larger `GEN_MAX_NEW_TOKENS` before anyone reads it as a finished summary), `known_prefix` (which trained prefix the input started with, or `None`), the `generation` settings actually used (`max_new_tokens`, `num_beams`, `do_sample=False`, `decision_rule`), `device`, `source` and the model identity. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind** — the counts above are counts, not scores; greedy decoding is an implicit per-step argmax with no minimum-probability cut-off, so some token is always produced, and the pipeline ships no acceptance threshold on output quality. Whoever deploys it owns any acceptance rule, judged on their own references. The run is deterministic for a given input, settings, weights, device and library versions (no sampling, `model.eval()`, no seed needed); float32 kernel differences between CPU and CUDA can flip a near-tied token and change the rest of the sequence from that point, and beam search can differ from greedy. The checks below are falsifiable plumbing checks — one result per input, every count within its ceiling, every default input recognised by its prefix — plus a per-call wall time measured on the runtime identified in Section 1 (the first call includes warm-up). Look for a short German sentence for `input00` and an English condensation for `input01`; whether they are *good* is exactly what no number here can tell you.

In [ ]:
import time

results = []
for item_id, text in zip(item_ids, texts, strict=True):
    started = time.perf_counter()
    result = pipe.generate(text, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    elapsed = time.perf_counter() - started
    results.append({'id': item_id, 'input': text, 'seconds': round(elapsed, 3), **result})
    print(f"{item_id} [{result['known_prefix'] or 'no known prefix'}] {result['input_tokens']} -> {result['generated_tokens']} tokens, stopped_by={result['stopped_by']}, {elapsed:.2f} s")
    print(f"    {result['text']}")
checks = {
    'one_result_per_input': len(results) == len(texts),
    'generated_within_ceiling': all(r['generated_tokens'] <= GEN_MAX_NEW_TOKENS for r in results),
    'input_within_ceiling': all(r['input_tokens'] <= MAX_INPUT_TOKENS for r in results),
    'settings_echoed': all(r['generation']['max_new_tokens'] == GEN_MAX_NEW_TOKENS and r['generation']['num_beams'] == NUM_BEAMS and r['generation']['do_sample'] is False for r in results),
}
if not USE_BYOD:
    checks['every_default_input_has_known_prefix'] = all(r['known_prefix'] is not None for r in results)
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
print({'checks': checks, 'decision_rule': results[0]['generation']['decision_rule'], 'hit_token_ceiling': [r['id'] for r in results if r['stopped_by'] == 'max_new_tokens']})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure**: summarisation and translation are conventionally scored with ROUGE-1/2/L and BLEU (or chrF) against human **reference outputs** — a reference summary per document, a reference translation per sentence — over enough items to state a dispersion, and the synthetic sample has none, so the verdict is always `not-measurable` and none is manufactured from a proxy such as length ratio or copy rate. Supplying a reference does not change the verdict, because no metric helper exists to score it and one reference is not a dispersion; the helper records that in `reason`. The report is written for the first generation and the verdict applies to every one of them equally — no metric exists for any. The upstream per-task figures in the paper's Table 14 are upstream claims, not measured here. It lands at `outputs/t5_small_text2text_evaluation_report.json`.

In [ ]:
report = evaluation_report(results[0], sample_kind=sample_kind)
with open('outputs/t5_small_text2text_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: the sample has no reference outputs, and the repository ships no metric helper; compute ROUGE/BLEU on your own referenced inputs.')

## 8. Export the generations and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report: `outputs/t5_small_text2text_generations.csv` — one row per input with its identifier, the known prefix, the input text, the generated text, both token counts, `stopped_by` and the wall time, so every generation maps back to its input — and `outputs/t5_small_text2text_result.json`, which carries the same items plus the generation settings in force, the ceilings, the sanity checks, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

items = [
    {'id': r['id'], 'known_prefix': r['known_prefix'], 'input': r['input'], 'output': r['text'], 'input_tokens': r['input_tokens'], 'generated_tokens': r['generated_tokens'], 'stopped_by': r['stopped_by'], 'seconds': r['seconds']}
    for r in results
]
with open('outputs/t5_small_text2text_generations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(items[0]))
    writer.writeheader()
    writer.writerows(items)
payload = {
    'items': items,
    'generation': results[0]['generation'],
    'ceilings': ceilings,
    'sanity_checks': checks,
    'generations_file': 'outputs/t5_small_text2text_generations.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
        'source': pipe.source,
    },
}
with open('outputs/t5_small_text2text_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The generated strings are the model's continuation of *your* prefixed input under greedy (or beam) decoding: fluent text that can add, drop or invert a fact, and the pipeline attaches no probability, confidence or quality score to it — `generated_tokens`, `input_tokens` and `stopped_by` are counts and flags, not evidence of correctness. On the synthetic sample the checks prove only that the input contract, the prefix convention, the verified snapshot load and the generation path work end to end; the evaluation report is `not-measurable` because none can be computed without reference outputs, and a real evaluation needs referenced inputs from your own domain, a ROUGE/BLEU-style scorer, and enough items to state a dispersion. An unknown prefix produces plausible-looking output rather than an error; inputs above `MAX_INPUT_TOKENS` are refused rather than cut; outputs that stop at `max_new_tokens` are truncated mid-thought. The pipeline exposes no instruction following, sampling, batching, or the upstream `task_specific_params` decoding settings. Decoding is deterministic on a fixed device and dtype, but CPU and CUDA float32 kernels can diverge on a near-tied token.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, summarisation or translation quality on any domain, a usable acceptance threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/t5-small/` and rerun Section 3. `ValueError: input is N tokens; ceiling is MAX_INPUT_TOKENS=512` in Section 6: split or shorten that BYOD line and rerun from Section 4. `stopped_by` equal to `max_new_tokens`: raise `GEN_MAX_NEW_TOKENS` (ceiling `MAX_NEW_TOKENS`) and rerun Section 6. Output that copies the input: the line has no trained prefix, or the passage is too short to summarise.

**Next experiments.** Set `NUM_BEAMS = 4` and compare the beam-search outputs with the greedy ones (the card-pass smoke found the short translation unchanged); translate the same sentence with the French and Romanian prefixes; hand the `summarize: ` input a passage you wrote a one-sentence reference summary for and score the output with a ROUGE implementation of your choice — the first step towards the real evaluation the report asks for; run the same batch on a CUDA runtime and diff the outputs against the CPU run. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/t5-small-text2text-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/t5-small-text2text-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/t5-small-text2text-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google-t5/t5-small
- Upstream code: https://github.com/google-research/text-to-text-transfer-transformer
- Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (Raffel et al., JMLR 2020): https://arxiv.org/abs/1910.10683